In [1]:
tabla='ctcam10'
col_tabla='citambproconfec'

In [2]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys

In [3]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [4]:

fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=2", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=2"

c1= text(query)
connection2.execute(c1)

2023-05-01


In [5]:

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

base2 = pd.read_sql_query(f"SELECT * FROM {tabla} WHERE {col_tabla} >= TO_DATE('{fecha}', 'YYYY-MM-DD')", con=connection0)




connection0.close()



In [6]:
base2.shape

(6291969, 38)

In [7]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6291969 entries, 0 to 6291968
Data columns (total 38 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   citamboricenasicod        object        
 1   citambcenasicod           object        
 2   citambnum                 int64         
 3   citambproconoricenasicod  object        
 4   citambproconcenasicod     object        
 5   citambarehoscod           object        
 6   citambservhoscod          object        
 7   citambactcod              object        
 8   citambactespcod           object        
 9   citambtipdocidenpercod    object        
 10  citambperasisdocidennum   object        
 11  citambproconfec           datetime64[ns]
 12  citambproconturhorini     object        
 13  citambproconturhorfin     object        
 14  tipocitacod               object        
 15  condcitacod               object        
 16  modotorcitacod            object        
 17  citambnu

In [8]:
base2.to_sql(name=f'tmp_{tabla}', con=engine3, if_exists='replace', index=False)

969

In [9]:


borrando=f"DELETE FROM {tabla} WHERE {col_tabla} >='{fecha}'"
borrado = connection3.execute(borrando)


query_count_before = f"SELECT COUNT(*) FROM {tabla}"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")


In [11]:


query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN citamboricenasicod TYPE character(1),
ALTER COLUMN citambcenasicod TYPE character(3),
ALTER COLUMN citambnum TYPE numeric(10,0) USING citambnum::numeric(10,0),
ALTER COLUMN citambproconoricenasicod TYPE character(1),
ALTER COLUMN citambproconcenasicod TYPE character(3),
ALTER COLUMN citambarehoscod TYPE character(2),
ALTER COLUMN citambservhoscod TYPE character(3),
ALTER COLUMN citambactcod TYPE character(2),
ALTER COLUMN citambactespcod TYPE character(3),
ALTER COLUMN citambtipdocidenpercod TYPE character(1),
ALTER COLUMN citambperasisdocidennum TYPE character(10),
ALTER COLUMN citambproconfec TYPE timestamp USING citambproconfec::timestamp without time zone,
ALTER COLUMN citambproconturhorini TYPE timestamp USING citambproconturhorini::timestamp without time zone,
ALTER COLUMN citambproconturhorfin TYPE timestamp USING citambproconturhorfin::timestamp without time zone,
ALTER COLUMN tipocitacod TYPE character(1),
ALTER COLUMN condcitacod TYPE character(1),
ALTER COLUMN modotorcitacod TYPE character(1),
ALTER COLUMN citambnumord TYPE numeric(3,0) USING citambnumord::numeric(3,0),
ALTER COLUMN citambhorcit TYPE timestamp USING citambhorcit::timestamp without time zone,
ALTER COLUMN citambrep TYPE character(1),
ALTER COLUMN motelicitcod TYPE character(2),
ALTER COLUMN estcitcod TYPE character(1),
ALTER COLUMN estcitotocod TYPE character(1),
ALTER COLUMN citambusucrecod TYPE character(10),
ALTER COLUMN citambcrefec TYPE timestamp USING citambcrefec::timestamp without time zone,
ALTER COLUMN citambusumodcod TYPE character(10),
ALTER COLUMN citambmodfec TYPE timestamp USING citambmodfec::timestamp without time zone, 
ALTER COLUMN citambsolfec TYPE timestamp USING citambsolfec::timestamp without time zone,
ALTER COLUMN citambipcrecod TYPE character(15),
ALTER COLUMN citambipmodcod TYPE character(15),
ALTER COLUMN motinacitdes TYPE character(110),
ALTER COLUMN citambpacsecnum TYPE numeric(10,0) USING citambpacsecnum::numeric(10,0),
ALTER COLUMN citambusuanucod TYPE character(10),
ALTER COLUMN citambanufec TYPE date USING citambanufec::date,
ALTER COLUMN citambipanucod TYPE character(15),
ALTER COLUMN citambmodanucod TYPE character(1),
ALTER COLUMN citambestctrlseg TYPE character(1),
ALTER COLUMN citambcnvespcod TYPE numeric(3,0) USING citambcnvespcod::numeric(3,0);


INSERT INTO {tabla} 
(citamboricenasicod,citambcenasicod,citambnum,citambproconoricenasicod,citambproconcenasicod,citambarehoscod,citambservhoscod,citambactcod,citambactespcod,citambtipdocidenpercod,citambperasisdocidennum,citambproconfec,citambproconturhorini,citambproconturhorfin,tipocitacod,condcitacod,modotorcitacod,citambnumord,citambhorcit,citambrep,motelicitcod,estcitcod,estcitotocod,citambusucrecod,citambcrefec,citambusumodcod,citambmodfec,citambsolfec,citambipcrecod,citambipmodcod,motinacitdes,citambpacsecnum,citambusuanucod,citambanufec,citambipanucod,citambmodanucod,citambestctrlseg,citambcnvespcod) 

SELECT 
citamboricenasicod,citambcenasicod,citambnum,citambproconoricenasicod,citambproconcenasicod,citambarehoscod,citambservhoscod,citambactcod,citambactespcod,citambtipdocidenpercod,citambperasisdocidennum,citambproconfec,citambproconturhorini,citambproconturhorfin,tipocitacod,condcitacod,modotorcitacod,citambnumord,citambhorcit,citambrep,motelicitcod,estcitcod,estcitotocod,citambusucrecod,citambcrefec,citambusumodcod,citambmodfec,citambsolfec,citambipcrecod,citambipmodcod,motinacitdes,citambpacsecnum,citambusuanucod,citambanufec,citambipanucod,citambmodanucod,citambestctrlseg,citambcnvespcod




FROM tmp_{tabla};
"""

c1= text(query)
connection3.execute(c1)


In [12]:

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
"""
c2= text(query2)
cursor=connection3.execute(c2)


#BORRAMOS LAS TABLAS
query2=f"""
SELECT COUNT(*) FROM {tabla} ;
"""
c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

connection3.close() 
